# 01 — Inspect Qwen3-0.6B KV Cache

**Project:** Dynamic Precision States for Paged KV Cache Management  
**Model:** `Qwen/Qwen3-0.6B`

## Goal
This notebook builds the full-precision baseline for the project. We load Qwen3-0.6B, run a forward pass with KV caching enabled, inspect the real cache structure, and measure its memory footprint.

### Questions
1. What cache implementation does Hugging Face return?
2. How many KV cache layers are stored?
3. What are the Key and Value tensor shapes and dtypes?
4. Which dimension represents sequence length?
5. How much memory does the KV cache use?


## 1. Environment Check

Verify the Python, PyTorch, Transformers, and CUDA environment before loading the model.


In [ ]:
import sys
import torch
import transformers

print("Python version:      ", sys.version.split()[0])
print("PyTorch version:     ", torch.__version__)
print("Transformers version:", transformers.__version__)
print("CUDA available:      ", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:                 ", torch.cuda.get_device_name(0))
    print("CUDA version:        ", torch.version.cuda)


: 

## 2. Imports and Configuration


In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen3-0.6B"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if DEVICE == "cuda":
    BF16_SUPPORTED = torch.cuda.is_bf16_supported()
    BF16_NATIVE = torch.cuda.is_bf16_supported(including_emulation=False)

    if BF16_NATIVE:
        MODEL_DTYPE = torch.bfloat16
    else:
        MODEL_DTYPE = torch.float16
else:
    BF16_SUPPORTED = False
    BF16_NATIVE = False
    MODEL_DTYPE = torch.float32

print(f"Model:                {MODEL_NAME}")
print(f"Device:               {DEVICE}")
print(f"BF16 supported:        {BF16_SUPPORTED}")
print(f"BF16 native support:   {BF16_NATIVE}")
print(f"Model dtype:           {MODEL_DTYPE}")


Model:                Qwen/Qwen3-0.6B
Device:               cuda
BF16 supported:        True
BF16 native support:   False
Model dtype:           torch.float16


## 3. Load Qwen3-0.6B

No KV quantization or paging changes are applied in this notebook. This is the baseline model.


In [8]:
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Clear previous CUDA allocations before loading the model
if DEVICE == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

# Load the baseline model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=MODEL_DTYPE,
)

model = model.to(DEVICE)
model.eval()

print("Model loaded successfully")
print(f"Model class:       {type(model).__name__}")
print(f"Vocabulary size:   {len(tokenizer):,}")

if DEVICE == "cuda":
    allocated_mb = torch.cuda.memory_allocated() / (1024 ** 2)
    reserved_mb = torch.cuda.memory_reserved() / (1024 ** 2)
    peak_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

    print(f"GPU allocated:     {allocated_mb:.2f} MB")
    print(f"GPU reserved:      {reserved_mb:.2f} MB")
    print(f"Peak GPU allocated:{peak_mb:.2f} MB")

Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 311/311 [00:01<00:00, 231.64it/s]


Model loaded successfully
Model class:       Qwen3ForCausalLM
Vocabulary size:   151,669
GPU allocated:     1136.89 MB
GPU reserved:      1160.00 MB
Peak GPU allocated:1136.89 MB


In [9]:
BASELINE_MODEL_MEMORY_MB = (
    torch.cuda.memory_allocated() / (1024 ** 2)
    if DEVICE == "cuda"
    else 0.0
)

print(f"\nBaseline model memory: {BASELINE_MODEL_MEMORY_MB:.2f} MB")


Baseline model memory: 1136.89 MB


## 4. Inspect Model Configuration

We read model parameters directly from the loaded configuration rather than assuming their values.


In [10]:
config = model.config

config_fields = {
    "Number of hidden layers": "num_hidden_layers",
    "Hidden size": "hidden_size",
    "Attention heads": "num_attention_heads",
    "KV heads": "num_key_value_heads",
    "Head dimension": "head_dim",
    "Maximum position embeddings": "max_position_embeddings",
}

print("=== QWEN3-0.6B MODEL CONFIGURATION ===")

for label, field in config_fields.items():
    value = getattr(config, field, "Not available")
    print(f"{label:30s}: {value}")

=== QWEN3-0.6B MODEL CONFIGURATION ===
Number of hidden layers       : 28
Hidden size                   : 1024
Attention heads               : 16
KV heads                      : 8
Head dimension                : 128
Maximum position embeddings   : 40960


## 5. Theoretical KV Cache Memory

The KV cache stores one Key tensor and one Value tensor for every Transformer layer and every cached token.

For a single token, the theoretical KV cache size is:

$$
2 \times N_{\text{layers}} \times H_{KV} \times D \times \text{bytes per element}
$$

where the factor 2 represents the Key and Value caches.

In [11]:
NUM_LAYERS = config.num_hidden_layers
NUM_KV_HEADS = config.num_key_value_heads
HEAD_DIM = config.head_dim
BYTES_PER_ELEMENT = torch.tensor([], dtype=MODEL_DTYPE).element_size()

kv_elements_per_token = (
    2
    * NUM_LAYERS
    * NUM_KV_HEADS
    * HEAD_DIM
)

kv_bytes_per_token = (
    kv_elements_per_token
    * BYTES_PER_ELEMENT
)

kv_kb_per_token = kv_bytes_per_token / 1024

print("=== THEORETICAL KV CACHE COST PER TOKEN ===")
print(f"Layers:              {NUM_LAYERS}")
print(f"KV heads:            {NUM_KV_HEADS}")
print(f"Head dimension:      {HEAD_DIM}")
print(f"Bytes per element:   {BYTES_PER_ELEMENT}")
print(f"KV elements/token:   {kv_elements_per_token:,}")
print(f"KV bytes/token:      {kv_bytes_per_token:,} bytes")
print(f"KV memory/token:     {kv_kb_per_token:.2f} KB")

=== THEORETICAL KV CACHE COST PER TOKEN ===
Layers:              28
KV heads:            8
Head dimension:      128
Bytes per element:   2
KV elements/token:   57,344
KV bytes/token:      114,688 bytes
KV memory/token:     112.00 KB


In [12]:
sequence_lengths = [
    128,
    256,
    512,
    1024,
    2048,
    4096,
    8192,
    16384,
    40960,
]

print("=== THEORETICAL FP16 KV CACHE MEMORY ===")

for seq_len in sequence_lengths:
    cache_mb = (
        kv_bytes_per_token
        * seq_len
        / (1024 ** 2)
    )

    print(
        f"Sequence length {seq_len:6d}: "
        f"{cache_mb:8.2f} MB"
    )

=== THEORETICAL FP16 KV CACHE MEMORY ===
Sequence length    128:    14.00 MB
Sequence length    256:    28.00 MB
Sequence length    512:    56.00 MB
Sequence length   1024:   112.00 MB
Sequence length   2048:   224.00 MB
Sequence length   4096:   448.00 MB
Sequence length   8192:   896.00 MB
Sequence length  16384:  1792.00 MB
Sequence length  40960:  4480.00 MB


## 6. Generate a Real KV Cache

We now run a forward pass with KV caching enabled.

The goal is to inspect the actual cache generated by Qwen3-0.6B and compare its tensor structure and memory footprint with the theoretical calculation above.

In [13]:
prompt = (
    "The KV cache stores Key and Value tensors during large language model "
    "inference. These tensors are reused during autoregressive decoding to "
    "avoid recomputing previous attention states."
)

inputs = tokenizer(
    prompt,
    return_tensors="pt",
)

inputs = {
    name: tensor.to(DEVICE)
    for name, tensor in inputs.items()
}

INPUT_SEQUENCE_LENGTH = inputs["input_ids"].shape[1]

print("=== INPUT ===")
print(f"Prompt:\n{prompt}")
print()
print(f"Input IDs shape:       {tuple(inputs['input_ids'].shape)}")
print(f"Input sequence length: {INPUT_SEQUENCE_LENGTH} tokens")

=== INPUT ===
Prompt:
The KV cache stores Key and Value tensors during large language model inference. These tensors are reused during autoregressive decoding to avoid recomputing previous attention states.

Input IDs shape:       (1, 32)
Input sequence length: 32 tokens


## 7. Run a Forward Pass with KV Caching

The `use_cache=True` option enables the KV cache.

After the forward pass, the cache is returned through `past_key_values`.

In [14]:
if DEVICE == "cuda":
    torch.cuda.reset_peak_memory_stats()

memory_before_forward_mb = (
    torch.cuda.memory_allocated() / (1024 ** 2)
    if DEVICE == "cuda"
    else 0.0
)

with torch.inference_mode():
    outputs = model(
        **inputs,
        use_cache=True,
        return_dict=True,
    )

cache = outputs.past_key_values

memory_after_forward_mb = (
    torch.cuda.memory_allocated() / (1024 ** 2)
    if DEVICE == "cuda"
    else 0.0
)

peak_memory_forward_mb = (
    torch.cuda.max_memory_allocated() / (1024 ** 2)
    if DEVICE == "cuda"
    else 0.0
)

print("=== FORWARD PASS ===")
print("Forward pass completed successfully")
print(f"Logits shape:          {tuple(outputs.logits.shape)}")
print(f"Cache type:            {type(cache).__name__}")
print(f"Number of cache layers:{len(cache)}")

if DEVICE == "cuda":
    print()
    print(f"Memory before forward: {memory_before_forward_mb:.2f} MB")
    print(f"Memory after forward:  {memory_after_forward_mb:.2f} MB")
    print(f"Peak memory:           {peak_memory_forward_mb:.2f} MB")

=== FORWARD PASS ===
Forward pass completed successfully
Logits shape:          (1, 32, 151936)
Cache type:            DynamicCache
Number of cache layers:28

Memory before forward: 1136.89 MB
Memory after forward:  1157.79 MB
Peak memory:           1157.85 MB


### GPU Memory Observation

The increase in GPU allocated memory after the forward pass is not an exact
measurement of the KV cache size. The forward pass also creates output tensors,
including the logits, and may keep additional CUDA allocations.

Therefore, the exact KV cache memory is measured directly from the Key and Value
tensors.

## 8. Inspect the KV Cache Structure

We now inspect the actual Key and Value tensors stored in the cache.

The goal is to identify the cache tensor dimensions and verify that they match
the model configuration.

In [15]:
print("=== CACHE OBJECT ===")
print(f"Cache class: {type(cache).__name__}")
print(f"Cache layers: {len(cache)}")

print("\nPublic cache attributes:")
print([
    name
    for name in dir(cache)
    if not name.startswith("_")
])

=== CACHE OBJECT ===
Cache class: DynamicCache
Cache layers: 28

Public cache attributes:
['batch_repeat_interleave', 'batch_select_indices', 'batch_size', 'crop', 'early_initialization', 'get_mask_sizes', 'get_max_cache_shape', 'get_max_length', 'get_seq_length', 'has_previous_state', 'is_compileable', 'is_initialized', 'is_linear', 'is_sliding', 'layer_class_to_replicate', 'layers', 'max_batch_size', 'max_cache_len', 'offload', 'offloading', 'prefetch', 'reorder_cache', 'reset', 'update', 'update_conv_state', 'update_indexer', 'update_recurrent_state']


In [16]:
print("=== CACHE STATE ===")

print(f"Initialized:        {cache.is_initialized}")
print(f"Sequence length:    {cache.get_seq_length()}")
print(f"Batch size:         {cache.batch_size}")
print(f"Number of layers:   {len(cache.layers)}")

=== CACHE STATE ===
Initialized:        True
Sequence length:    32
Batch size:         -1
Number of layers:   28


### Inspect a Single Cache Layer

Each Transformer layer stores its own Key and Value tensors.

We inspect the first cache layer to identify the tensor layout used by Qwen3-0.6B.

In [17]:
layer0 = cache.layers[0]

print("=== CACHE LAYER 0 ===")

print(f"Layer class: {type(layer0).__name__}")

print("\nPublic layer attributes:")
print([
    name
    for name in dir(layer0)
    if not name.startswith("_")
])

=== CACHE LAYER 0 ===
Layer class: DynamicLayer

Public layer attributes:
['batch_repeat_interleave', 'batch_select_indices', 'crop', 'device', 'dtype', 'get_mask_sizes', 'get_max_cache_shape', 'get_max_length', 'get_seq_length', 'is_compileable', 'is_initialized', 'is_sliding', 'keys', 'layer_type', 'lazy_initialization', 'offload', 'prefetch', 'reorder_cache', 'reset', 'supports_early_init', 'update', 'values']


### Key and Value Tensor Layout

The first cache layer exposes the stored Key and Value tensors directly.

We inspect their shapes to identify the batch, KV head, sequence, and head
dimension layout used by Qwen3-0.6B.


In [18]:
key0 = layer0.keys
value0 = layer0.values

print("=== KEY AND VALUE TENSORS — LAYER 0 ===")

print(f"Key type:       {type(key0).__name__}")
print(f"Value type:     {type(value0).__name__}")

print()
print(f"Key shape:      {tuple(key0.shape)}")
print(f"Value shape:    {tuple(value0.shape)}")

print()
print(f"Key dtype:      {key0.dtype}")
print(f"Value dtype:    {value0.dtype}")

print()
print(f"Key device:     {key0.device}")
print(f"Value device:   {value0.device}")

print()
print(f"Key elements:   {key0.numel():,}")
print(f"Value elements: {value0.numel():,}")

=== KEY AND VALUE TENSORS — LAYER 0 ===
Key type:       Tensor
Value type:     Tensor

Key shape:      (1, 8, 32, 128)
Value shape:    (1, 8, 32, 128)

Key dtype:      torch.float16
Value dtype:    torch.float16

Key device:     cuda:0
Value device:   cuda:0

Key elements:   32,768
Value elements: 32,768


## 9. Exact KV Cache Memory Measurement

The exact KV cache memory is measured directly from the stored Key and Value
tensors.

For each tensor, the memory footprint is calculated as:

$$
\text{Tensor Memory}
=
\text{Number of Elements}
\times
\text{Bytes per Element}
$$

We sum the Key and Value tensor sizes across all Transformer layers.

In [19]:
def tensor_size_bytes(tensor: torch.Tensor) -> int:
    return tensor.numel() * tensor.element_size()


total_key_bytes = 0
total_value_bytes = 0

for layer in cache.layers:
    total_key_bytes += tensor_size_bytes(layer.keys)
    total_value_bytes += tensor_size_bytes(layer.values)

total_cache_bytes = total_key_bytes + total_value_bytes

KB = 1024
MB = 1024 ** 2

print("=== EXACT KV CACHE MEMORY ===")

print(f"Key cache memory:     {total_key_bytes / MB:.4f} MB")
print(f"Value cache memory:   {total_value_bytes / MB:.4f} MB")
print(f"Total KV cache:       {total_cache_bytes / MB:.4f} MB")

print()
print(f"Total KV cache:       {total_cache_bytes / KB:.2f} KB")
print(f"Bytes per token:      {total_cache_bytes / INPUT_SEQUENCE_LENGTH:,.0f}")
print(f"KB per token:         {total_cache_bytes / INPUT_SEQUENCE_LENGTH / KB:.2f}")

=== EXACT KV CACHE MEMORY ===
Key cache memory:     1.7500 MB
Value cache memory:   1.7500 MB
Total KV cache:       3.5000 MB

Total KV cache:       3584.00 KB
Bytes per token:      114,688
KB per token:         112.00


In [20]:
theoretical_cache_bytes = (
    kv_bytes_per_token
    * INPUT_SEQUENCE_LENGTH
)

print("=== THEORETICAL VS MEASURED ===")

print(
    f"Theoretical KV cache: "
    f"{theoretical_cache_bytes / MB:.4f} MB"
)

print(
    f"Measured KV cache:    "
    f"{total_cache_bytes / MB:.4f} MB"
)

difference_bytes = (
    total_cache_bytes
    - theoretical_cache_bytes
)

print(
    f"Difference:           "
    f"{difference_bytes:,} bytes"
)

assert total_cache_bytes == theoretical_cache_bytes

print()
print("Theoretical and measured KV cache sizes match exactly.")

=== THEORETICAL VS MEASURED ===
Theoretical KV cache: 3.5000 MB
Measured KV cache:    3.5000 MB
Difference:           0 bytes

Theoretical and measured KV cache sizes match exactly.


### Observation

The measured KV cache size exactly matches the theoretical calculation.

For Qwen3-0.6B in FP16, the KV cache requires **112 KB per cached token** for
batch size 1. Therefore, the KV cache memory grows linearly with sequence length.

This confirms that reducing the precision of KV cache pages can directly reduce
the memory required per cached token.

## 10. Divide the KV Cache into Fixed-Size Pages

Our project studies precision management at the KV page level.

Since the sequence dimension of the Qwen3-0.6B KV cache is dimension 2, we can
divide the cached tokens into fixed-size pages along this dimension.

In this initial experiment, we use a page size of 8 tokens.

In [21]:
PAGE_SIZE = 8

sequence_length = key0.shape[2]

num_pages = (
    sequence_length + PAGE_SIZE - 1
) // PAGE_SIZE

print("=== KV PAGE CONFIGURATION ===")

print(f"Sequence length: {sequence_length}")
print(f"Page size:       {PAGE_SIZE} tokens")
print(f"Number of pages: {num_pages}")

=== KV PAGE CONFIGURATION ===
Sequence length: 32
Page size:       8 tokens
Number of pages: 4


In [22]:
kv_pages = []

for page_id in range(num_pages):
    start_token = page_id * PAGE_SIZE
    end_token = min(
        start_token + PAGE_SIZE,
        sequence_length,
    )

    page = {
        "page_id": page_id,
        "start_token": start_token,
        "end_token": end_token,
        "num_tokens": end_token - start_token,
    }

    kv_pages.append(page)

print("=== KV PAGE METADATA ===")

for page in kv_pages:
    print(
        f"Page {page['page_id']}: "
        f"tokens [{page['start_token']}, "
        f"{page['end_token'] - 1}] "
        f"({page['num_tokens']} tokens)"
    )

=== KV PAGE METADATA ===
Page 0: tokens [0, 7] (8 tokens)
Page 1: tokens [8, 15] (8 tokens)
Page 2: tokens [16, 23] (8 tokens)
Page 3: tokens [24, 31] (8 tokens)


In [23]:
layer0_key_pages = []
layer0_value_pages = []

for page in kv_pages:
    start = page["start_token"]
    end = page["end_token"]

    key_page = key0[:, :, start:end, :]
    value_page = value0[:, :, start:end, :]

    layer0_key_pages.append(key_page)
    layer0_value_pages.append(value_page)

print("=== LAYER 0 PAGE TENSORS ===")

for page_id, (key_page, value_page) in enumerate(
    zip(layer0_key_pages, layer0_value_pages)
):
    print(f"\nPage {page_id}")
    print(f"Key shape:   {tuple(key_page.shape)}")
    print(f"Value shape: {tuple(value_page.shape)}")

=== LAYER 0 PAGE TENSORS ===

Page 0
Key shape:   (1, 8, 8, 128)
Value shape: (1, 8, 8, 128)

Page 1
Key shape:   (1, 8, 8, 128)
Value shape: (1, 8, 8, 128)

Page 2
Key shape:   (1, 8, 8, 128)
Value shape: (1, 8, 8, 128)

Page 3
Key shape:   (1, 8, 8, 128)
Value shape: (1, 8, 8, 128)


## 11. Add Precision State to KV Pages

Each logical KV page is associated with metadata used by the precision management
policy.

The initial metadata contains:

- a page identifier,
- the token range stored in the page,
- the current precision state,
- the page age,
- an importance score.

At this stage, the metadata is only a logical management structure. The underlying
KV tensors are still stored in FP16.


In [24]:
PRECISION_FP16 = "FP16"
PRECISION_INT4 = "INT4"
PRECISION_INT2 = "INT2"

for page in kv_pages:
    page["precision_state"] = PRECISION_FP16
    page["age"] = 0
    page["importance_score"] = 0.0

print("=== INITIAL KV PAGE STATES ===")

for page in kv_pages:
    print(
        f"Page {page['page_id']}: "
        f"tokens [{page['start_token']}, {page['end_token'] - 1}] | "
        f"precision={page['precision_state']} | "
        f"age={page['age']} | "
        f"importance={page['importance_score']:.2f}"
    )

=== INITIAL KV PAGE STATES ===
Page 0: tokens [0, 7] | precision=FP16 | age=0 | importance=0.00
Page 1: tokens [8, 15] | precision=FP16 | age=0 | importance=0.00
Page 2: tokens [16, 23] | precision=FP16 | age=0 | importance=0.00
Page 3: tokens [24, 31] | precision=FP16 | age=0 | importance=0.00


### Page Age

For an initial recency-based policy, page age is defined as the number of pages
between the current page and the newest cached page.

The newest page has age 0.

In [25]:
newest_page_id = num_pages - 1

for page in kv_pages:
    page["age"] = newest_page_id - page["page_id"]

print("=== KV PAGE AGE ===")

for page in kv_pages:
    print(
        f"Page {page['page_id']}: "
        f"age={page['age']}"
    )

=== KV PAGE AGE ===
Page 0: age=3
Page 1: age=2
Page 2: age=1
Page 3: age=0


### Initial Recency-Based Precision Policy

We first define a simple recency-based policy.

Recent pages are assigned higher precision, while older pages are assigned lower
precision.

This policy is intentionally simple and will later be compared with fixed
quantization and attention-based importance policies.

In [26]:
def assign_recency_precision(page_age: int) -> str:
    if page_age == 0:
        return PRECISION_FP16

    if page_age == 1:
        return PRECISION_INT4

    return PRECISION_INT2


for page in kv_pages:
    page["precision_state"] = assign_recency_precision(
        page["age"]
    )

print("=== RECENCY-BASED PAGE PRECISION ===")

for page in kv_pages:
    print(
        f"Page {page['page_id']}: "
        f"age={page['age']} | "
        f"precision={page['precision_state']}"
    )

=== RECENCY-BASED PAGE PRECISION ===
Page 0: age=3 | precision=INT2
Page 1: age=2 | precision=INT2
Page 2: age=1 | precision=INT4
Page 3: age=0 | precision=FP16


## Next Step

After validating this baseline, the next notebook will measure KV cache growth for multiple sequence lengths and analyze Key/Value value distributions before implementing quantization.
